In [1]:
# =========================================================
# PS S6E4 - exp_20260410_039_xgb_formula_logit_min
# Traiko-style formula/logit features as an independent lightweight family
# raw base 19 + threshold 4 + precomputed logits 3
# no pairwise / no pseudo / no posthoc / no original merge
# =========================================================

## Import / config

In [2]:
import os
import gc
import json
import random

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

# ---------------------------------------------------------
# Config
# ---------------------------------------------------------
class CFG:
    EXP_ID = "exp_20260410_039_xgb_formula_logit_min"
    SEED = 42
    N_SPLITS = 5

    TARGET = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    ORIG_PATH = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    OUTPUT_DIR = f"/kaggle/working/{EXP_ID}"

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    # threshold features
    SOIL_THRESH = 25
    TEMP_THRESH = 30
    RAIN_THRESH = 300
    WIND_THRESH = 10

    # precomputed logits
    LOGIT_COEFS = {
        "Low": {
            "intercept": 16.3173,
            "soil_lt_25": -11.0237,
            "temp_gt_30": -5.8559,
            "rain_lt_300": -10.8500,
            "wind_gt_10": -5.8284,
            "Flowering": -5.4155,
            "Harvest": 5.5073,
            "Sowing": 5.2299,
            "Vegetative": -5.4617,
            "Mulch_No": -3.0014,
            "Mulch_Yes": 2.8613,
        },
        "Medium": {
            "intercept": 4.6524,
            "soil_lt_25": 0.3290,
            "temp_gt_30": -0.0204,
            "rain_lt_300": 0.1542,
            "wind_gt_10": 0.0841,
            "Flowering": 0.3586,
            "Harvest": -0.1348,
            "Sowing": -0.3547,
            "Vegetative": 0.3334,
            "Mulch_No": 0.1883,
            "Mulch_Yes": 0.0142,
        },
        "High": {
            "intercept": -20.9697,
            "soil_lt_25": 10.6947,
            "temp_gt_30": 5.8763,
            "rain_lt_300": 10.6958,
            "wind_gt_10": 5.7444,
            "Flowering": 5.0569,
            "Harvest": -5.3725,
            "Sowing": -4.8752,
            "Vegetative": 5.1283,
            "Mulch_No": 2.8131,
            "Mulch_Yes": -2.8755,
        },
    }

    XGB_PARAMS = {
        "n_estimators": 5000,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 2.0,
        "subsample": 0.85,
        "colsample_bytree": 0.60,
        "gamma": 1.0,
        "reg_alpha": 0.01,
        "reg_lambda": 1.0,
        "objective": "multi:softprob",
        "num_class": 3,
        "tree_method": "hist",
        "enable_categorical": False,
        "eval_metric": "mlogloss",
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": 0,
    }

## Utils

In [3]:
# ---------------------------------------------------------
# Utils
# ---------------------------------------------------------
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

def save_json(obj, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def rankdata_1d(x: np.ndarray) -> np.ndarray:
    return pd.Series(x).rank(method="average").to_numpy()

def mean_raw_corr_multiclass(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))

def mean_rank_corr_multiclass(a: np.ndarray, b: np.ndarray) -> float:
    vals = []
    for c in range(a.shape[1]):
        ra = rankdata_1d(a[:, c])
        rb = rankdata_1d(b[:, c])
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))

def add_formula_logit_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["soil_lt_25"] = (out["Soil_Moisture"] < CFG.SOIL_THRESH).astype("int8")
    out["temp_gt_30"] = (out["Temperature_C"] > CFG.TEMP_THRESH).astype("int8")
    out["rain_lt_300"] = (out["Rainfall_mm"] < CFG.RAIN_THRESH).astype("int8")
    out["wind_gt_10"] = (out["Wind_Speed_kmh"] > CFG.WIND_THRESH).astype("int8")

    stage_flowering = (out["Crop_Growth_Stage"] == "Flowering").astype("int8")
    stage_harvest = (out["Crop_Growth_Stage"] == "Harvest").astype("int8")
    stage_sowing = (out["Crop_Growth_Stage"] == "Sowing").astype("int8")
    stage_vegetative = (out["Crop_Growth_Stage"] == "Vegetative").astype("int8")

    mulch_no = (out["Mulching_Used"] == "No").astype("int8")
    mulch_yes = (out["Mulching_Used"] == "Yes").astype("int8")

    for cls_name in ["Low", "Medium", "High"]:
        coef = CFG.LOGIT_COEFS[cls_name]

        out[f"logit(P(y={cls_name}))"] = (
            coef["intercept"]
            + coef["soil_lt_25"] * out["soil_lt_25"]
            + coef["temp_gt_30"] * out["temp_gt_30"]
            + coef["rain_lt_300"] * out["rain_lt_300"]
            + coef["wind_gt_10"] * out["wind_gt_10"]
            + coef["Flowering"] * stage_flowering
            + coef["Harvest"] * stage_harvest
            + coef["Sowing"] * stage_sowing
            + coef["Vegetative"] * stage_vegetative
            + coef["Mulch_No"] * mulch_no
            + coef["Mulch_Yes"] * mulch_yes
        ).astype("float32")

    return out

## Load data & FE

In [4]:
# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
seed_everything(CFG.SEED)
ensure_dir(CFG.OUTPUT_DIR)

train = pd.read_csv(CFG.TRAIN_PATH)
test = pd.read_csv(CFG.TEST_PATH)
orig = pd.read_csv(CFG.ORIG_PATH)  # loaded for consistency only

train[CFG.TARGET] = train[CFG.TARGET].map(CFG.TARGET_MAP)

num_features = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

cat_features = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region",
]

raw_base_cols = num_features + cat_features

train_eng = train[[CFG.ID_COL] + raw_base_cols + [CFG.TARGET]].copy()
test_eng = test[[CFG.ID_COL] + raw_base_cols].copy()

train_eng = add_formula_logit_features(train_eng)
test_eng = add_formula_logit_features(test_eng)

threshold_cols = [
    "soil_lt_25",
    "temp_gt_30",
    "rain_lt_300",
    "wind_gt_10",
]

logit_cols = [
    "logit(P(y=Low))",
    "logit(P(y=Medium))",
    "logit(P(y=High))",
]

feature_cols = raw_base_cols + threshold_cols + logit_cols

X_train = train_eng[feature_cols].copy()
X_test = test_eng[feature_cols].copy()
y = train_eng[CFG.TARGET].values

# joint factorization for categoricals
for c in cat_features:
    combined = pd.concat([X_train[c].astype(str), X_test[c].astype(str)], axis=0, ignore_index=True)
    codes, _ = pd.factorize(combined)
    X_train[c] = codes[:len(X_train)].astype("float32")
    X_test[c] = codes[len(X_train):].astype("float32")

# numeric cast / fill
for c in num_features + threshold_cols + logit_cols:
    med = X_train[c].median()
    X_train[c] = X_train[c].fillna(med).astype("float32")
    X_test[c] = X_test[c].fillna(med).astype("float32")

## CV train

In [5]:
# ---------------------------------------------------------
# CV training
# ---------------------------------------------------------
skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED
)

oof_probs = np.zeros((len(X_train), 3), dtype=np.float32)
test_probs = np.zeros((len(X_test), 3), dtype=np.float32)

fold_scores = []
best_iterations = []
feature_importance_list = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y), 1):
    print(f"\n===== Fold {fold}/{CFG.N_SPLITS} =====")

    Xt = X_train.iloc[tr_idx].reset_index(drop=True)
    Xv = X_train.iloc[va_idx].reset_index(drop=True)
    Xe = X_test.reset_index(drop=True)

    yt = y[tr_idx]
    yv = y[va_idx]

    sw = compute_sample_weight("balanced", yt)

    model = xgb.XGBClassifier(
        **CFG.XGB_PARAMS,
        early_stopping_rounds=200
    )

    model.fit(
        Xt,
        yt,
        sample_weight=sw,
        eval_set=[(Xv, yv)],
        verbose=200
    )

    oof_probs[va_idx] = model.predict_proba(Xv)
    test_probs += model.predict_proba(Xe) / CFG.N_SPLITS

    fold_score = balanced_accuracy_score(yv, np.argmax(oof_probs[va_idx], axis=1))
    fold_scores.append(float(fold_score))
    best_iterations.append(int(getattr(model, "best_iteration", model.n_estimators)))

    booster_score = model.get_booster().get_score(importance_type="gain")
    feature_importance_list.append(booster_score)

    print(f"Fold {fold} BA: {fold_score:.6f}")

    del Xt, Xv, Xe, yt, yv, sw, model
    gc.collect()

cv_score = float(balanced_accuracy_score(y, np.argmax(oof_probs, axis=1)))
print(f"\nOOF Balanced Accuracy: {cv_score:.6f}")


===== Fold 1/5 =====
[0]	validation_0-mlogloss:1.07061
[200]	validation_0-mlogloss:0.08100
[400]	validation_0-mlogloss:0.07372
[600]	validation_0-mlogloss:0.07045
[800]	validation_0-mlogloss:0.06818
[1000]	validation_0-mlogloss:0.06643
[1200]	validation_0-mlogloss:0.06503
[1400]	validation_0-mlogloss:0.06392
[1600]	validation_0-mlogloss:0.06297
[1800]	validation_0-mlogloss:0.06209
[2000]	validation_0-mlogloss:0.06133
[2200]	validation_0-mlogloss:0.06070
[2400]	validation_0-mlogloss:0.06015
[2600]	validation_0-mlogloss:0.05964
[2800]	validation_0-mlogloss:0.05920
[3000]	validation_0-mlogloss:0.05881
[3200]	validation_0-mlogloss:0.05848
[3400]	validation_0-mlogloss:0.05818
[3600]	validation_0-mlogloss:0.05790
[3800]	validation_0-mlogloss:0.05766
[4000]	validation_0-mlogloss:0.05744
[4200]	validation_0-mlogloss:0.05726
[4400]	validation_0-mlogloss:0.05711
[4600]	validation_0-mlogloss:0.05696
[4800]	validation_0-mlogloss:0.05684
[4999]	validation_0-mlogloss:0.05674
Fold 1 BA: 0.970157

==

## Save outputs

In [6]:
# ---------------------------------------------------------
# Save outputs
# ---------------------------------------------------------
np.save(f"{CFG.OUTPUT_DIR}/oof_xgb_formula_logit_min_proba.npy", oof_probs)
np.save(f"{CFG.OUTPUT_DIR}/pred_xgb_formula_logit_min_proba.npy", test_probs)

submission = pd.DataFrame({
    CFG.ID_COL: test[CFG.ID_COL],
    CFG.TARGET: pd.Series(np.argmax(test_probs, axis=1)).map(CFG.INV_TARGET_MAP)
})
submission.to_csv(f"{CFG.OUTPUT_DIR}/submission_xgb_formula_logit_min.csv", index=False)

feature_columns = pd.DataFrame({
    "feature_name": feature_cols,
    "feature_group": (
        ["raw_base"] * len(raw_base_cols)
        + ["threshold"] * len(threshold_cols)
        + ["precomputed_logit"] * len(logit_cols)
    )
})
feature_columns.to_csv(f"{CFG.OUTPUT_DIR}/feature_columns.csv", index=False)

all_feats = set()
for d in feature_importance_list:
    all_feats.update(d.keys())

rows = []
for feat in sorted(all_feats):
    vals = [imp.get(feat, 0.0) for imp in feature_importance_list]
    rows.append({
        "feature_name": feat,
        "importance_gain_mean": float(np.mean(vals)),
        "importance_gain_std": float(np.std(vals)),
    })

feature_importance_df = pd.DataFrame(rows).sort_values(
    "importance_gain_mean", ascending=False
)
feature_importance_df.to_csv(f"{CFG.OUTPUT_DIR}/feature_importance.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "model": "XGBoost",
    "metric": "balanced_accuracy_score",
    "seed": CFG.SEED,
    "fold_type": "StratifiedKFold",
    "n_splits": CFG.N_SPLITS,
    "raw_base_feature_count": len(raw_base_cols),
    "threshold_feature_count": len(threshold_cols),
    "precomputed_logit_feature_count": len(logit_cols),
    "fold_scores": fold_scores,
    "cv_score_raw": cv_score,
    "best_iterations": best_iterations,
    "xgb_params": CFG.XGB_PARAMS,
    "notes": {
        "pairwise": False,
        "pseudo_label": False,
        "posthoc_optimization": False,
        "original_merge": False,
        "digit_features": False,
        "ordered_te": False,
        "formula_thresholds": True,
        "precomputed_logits": True,
    }
}
save_json(cv_result, f"{CFG.OUTPUT_DIR}/cv_result.json")

print("\nSaved files:")
for fn in [
    "oof_xgb_formula_logit_min_proba.npy",
    "pred_xgb_formula_logit_min_proba.npy",
    "submission_xgb_formula_logit_min.csv",
    "feature_columns.csv",
    "feature_importance.csv",
    "cv_result.json",
]:
    print("-", f"{CFG.OUTPUT_DIR}/{fn}")


Saved files:
- /kaggle/working/exp_20260410_039_xgb_formula_logit_min/oof_xgb_formula_logit_min_proba.npy
- /kaggle/working/exp_20260410_039_xgb_formula_logit_min/pred_xgb_formula_logit_min_proba.npy
- /kaggle/working/exp_20260410_039_xgb_formula_logit_min/submission_xgb_formula_logit_min.csv
- /kaggle/working/exp_20260410_039_xgb_formula_logit_min/feature_columns.csv
- /kaggle/working/exp_20260410_039_xgb_formula_logit_min/feature_importance.csv
- /kaggle/working/exp_20260410_039_xgb_formula_logit_min/cv_result.json
